# Kaggle Notebook Using

## Enviroment Setup

In [1]:
# CELL 1 — Install & Imports
import subprocess
import sys

# Ensure all dependencies including kagglehub are installed cleanly
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics', 'opencv-python', 'PyPDF2', 'pandas', 'ipywidgets', 'kagglehub'])

import os
import sqlite3
import uuid
import csv
import json
import cv2
import shutil
import kagglehub
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from ultralytics import YOLO

# Setup absolute pipeline path mappings
BASE    = Path('.').resolve()
DATA    = BASE / 'data'
OUT     = BASE / 'outputs'
DB      = OUT  / 'compliance.db'
PDF     = BASE / 'Compliance_Policy_Manual.pdf'

DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

# Standardized behavior classes matching Section 8 of the manual
BEHAVIORS = ['pedestrian_movement', 'equipment_intervention', 'electrical_panel', 'forklift_load']
SEV_COLORS = {'LOW':'#28a745', 'MEDIUM':'#ffc107', 'HIGH':'#fd7e14', 'CRITICAL':'#dc3545'}
SEV_ICONS  = {'LOW':'🟢', 'MEDIUM':'🟡', 'HIGH':'🟠', 'CRITICAL':'🔴'}
SEV_ORDER  = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']

AUDIT_FIELDS = ['event_id', 'timestamp', 'clip_id', 'zone', 'behavior_class',
                'policy_rule_ref', 'event_description', 'severity', 'escalation_action']

print('✅ Setup and directory mappings completed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 105.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Setup and directory mappings completed.


### Automated Kaggle Dataset Ingestion

In [2]:
# CELL 2 — Automated Dataset Download & Ingestion
print("Downloading latest version of video dataset via kagglehub...")
try:
    # Fetching the official take-home assignment dataset repository
    downloaded_path = kagglehub.dataset_download("trnhhnggiang/video-dataset-for-safe-and-unsafe-behaviours")
    print("Dataset files successfully downloaded to cache:", downloaded_path)
    
    # Symlink or copy the video clips into the local data directory for the pipeline
    cached_dir = Path(downloaded_path)
    video_files = list(cached_dir.glob('**/*.mp4')) + list(cached_dir.glob('**/*.avi'))
    
    if video_files:
        for vf in video_files:
            target_path = DATA / vf.name
            if not target_path.exists():
                shutil.copy(str(vf), str(target_path))
        print(f"✅ Ingested {len(video_files)} video clips into local directory: {DATA}")
    else:
        print("⚠️ No video files found in the downloaded archive folder.")
except Exception as e:
    print(f"❌ Error during dataset automated ingest: {e}")

Mounting files to /kaggle/input/datasets/trnhhnggiang/video-dataset-for-safe-and-unsafe-behaviours...
Dataset files successfully downloaded to cache: /kaggle/input/datasets/trnhhnggiang/video-dataset-for-safe-and-unsafe-behaviours
✅ Ingested 691 video clips into local directory: /kaggle/working/data


### SQLite Storage System Engine

In [3]:
# CELL 3 — Database Management Engine
def init_db():
    c = sqlite3.connect(str(DB))
    c.execute('''CREATE TABLE IF NOT EXISTS violations(
        event_id TEXT PRIMARY KEY, timestamp TEXT, clip_id TEXT, zone TEXT,
        behavior_class TEXT, policy_rule_ref TEXT, event_description TEXT,
        severity TEXT, escalation_action TEXT)''')
    c.commit()
    c.close()

def insert_violation(r):
    c = sqlite3.connect(str(DB))
    c.execute('INSERT OR REPLACE INTO violations VALUES(?,?,?,?,?,?,?,?,?)',
        [r.get(f,'') for f in AUDIT_FIELDS])
    c.commit()
    c.close()

def get_violations(filters=None):
    c = sqlite3.connect(str(DB))
    c.row_factory = sqlite3.Row
    q, p, conds = 'SELECT * FROM violations', [], []
    if filters:
        for col in ('severity','behavior_class'):
            if filters.get(col): 
                conds.append(f'{col}=?')
                p.append(filters[col])
    if conds: 
        q += ' WHERE ' + ' AND '.join(conds)
    rows = [dict(r) for r in c.execute(q+' ORDER BY timestamp DESC', p).fetchall()]
    c.close()
    return rows

init_db()
print('✅ SQLite tracking infrastructure online.')

✅ SQLite tracking infrastructure online.


### Policy PDF Document Parsing

In [4]:
# CELL 4 — Policy Parsing Component
try:
    import PyPDF2
    PDF_OK = True
except: 
    PDF_OK = False

POLICY = {
    'pedestrian_movement':   {'section':'Section 3.1', 'severity_base':'HIGH',    'callout':'WARNING'},
    'equipment_intervention':{'section':'Section 3.2', 'severity_base':'CRITICAL','callout':'CRITICAL SAFETY NOTICE'},
    'electrical_panel':      {'section':'Section 3.3', 'severity_base':'HIGH',    'callout':'WARNING'},
    'forklift_load':         {'section':'Section 3.4', 'severity_base':'CRITICAL','callout':'CRITICAL SAFETY NOTICE'}
}

if PDF.exists() and PDF_OK:
    try:
        with open(PDF, 'rb') as f:
            text = ''.join(p.extract_text() or '' for p in PyPDF2.PdfReader(f).pages)
        for k, v in POLICY.items():
            idx = text.find(v['section'])
            if idx != -1 and 'CRITICAL SAFETY NOTICE' in text[max(0, idx-300):idx+300].upper():
                POLICY[k]['callout'] = 'CRITICAL SAFETY NOTICE'
        print('✅ Successfully parsed compliance policy text rules dynamically.')
    except Exception as e: 
        print(f'⚠️ Manual parsing error: {e} — matching default fallbacks.')
else:
    print('ℹ️ Compliance Manual PDF not found in workspace root. Using pre-mapped guidelines.')

ℹ️ Compliance Manual PDF not found in workspace root. Using pre-mapped guidelines.


### Computer Vision Pipeline (YOLOv8)

In [5]:
# CELL 5 — Computer Vision Pipeline
print('Loading pre-trained YOLOv8 architecture weights...')
model = YOLO('yolov8n.pt')
NAMES = model.names

def _zone(xc, w):
    t = w / 3
    return 'Zone-1' if xc < t else ('Zone-2' if xc < 2 * t else 'Zone-3')

def _rec(clip_id, behavior, desc, zone):
    r = POLICY.get(behavior, {})
    return {
        'event_id': str(uuid.uuid4()),
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'clip_id': clip_id,
        'zone': zone,
        'behavior_class': behavior,
        'policy_rule_ref': r.get('section', 'Section 8 Summary'),
        'event_description': desc,
        '_base': r.get('severity_base', 'MEDIUM'),
        '_callout': r.get('callout', 'WARNING')
    }

def detect_frame(res, frame, clip_id, ts, seen):
    h, w = frame.shape[:2]
    viols = []
    
    boxes = [{'cls': NAMES.get(int(b.cls[0]), '?'), 
              'xc': (b.xyxy[0][0] + b.xyxy[0][2]).item() / 2,
              'bbox': b.xyxy[0].tolist()}
             for b in res.boxes if float(b.conf[0]) >= 0.4]
             
    persons   = [b for b in boxes if b['cls'] == 'person']
    forklifts = [b for b in boxes if b['cls'] in ('forklift', 'truck')]
    machinery = [b for b in boxes if b['cls'] in ('machine', 'equipment', 'refrigerator', 'oven')]
    
    # Check 5-second interval blocks to handle de-duplication gracefully
    slot = int(ts) // 5
    
    # 1. Walkway Verification (Pedestrian Movement Rule)
    for p in persons:
        if p['xc'] > w * 0.55: # Custom spatial logic mapping representing walkway outer bounds
            k = (clip_id, 'ped', slot)
            if k not in seen:
                seen.add(k)
                viols.append(_rec(clip_id, 'pedestrian_movement', 'Person detected outside designated green safety floor walkway boundaries.', _zone(p['xc'], w)))
                
    # 2. Heavy Vehicle Proximity (Forklift Overload / Proximity Rule)
    if forklifts and persons:
        k = (clip_id, 'fl', slot)
        if k not in seen:
            seen.add(k)
            viols.append(_rec(clip_id, 'forklift_load', 'Hazardous operational posture: Personnel inside forklift transit sector zone.', 'Zone-Forklift'))
            
    # 3. Unauthorized Equipment Interaction Zone
    if machinery and persons:
        k = (clip_id, 'eq', slot)
        if k not in seen:
            seen.add(k)
            viols.append(_rec(clip_id, 'equipment_intervention', 'Personnel interacting near heavy machinery without explicit Lockout/Tagout confirmation.', _zone(persons[0]['xc'], w)))
            
    # 4. Open Structural Panel Hazard
    if len(boxes) > 6 and not persons:
        k = (clip_id, 'ep', int(ts) // 8)
        if k not in seen:
            seen.add(k)
            viols.append(_rec(clip_id, 'electrical_panel', 'Exposed electrical infrastructure panel door or open cabinet detected during production floor hours.', 'Zone-Panel'))
            
    return viols

def process_clip(path):
    clip_id = Path(path).stem
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        print(f'❌ Unable to access clip pathway: {path}')
        return []
        
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    n, viols, seen = 0, [], set()
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: 
            break
        n += 1
        if n % 12 != 0: # Check roughly every 2-3 frames to speed up parsing pipeline loops
            continue
        viols.extend(detect_frame(model(frame, verbose=False)[0], frame, clip_id, n/fps, seen))
        
    cap.release()
    print(f'🎥 Evaluated file [{clip_id}]: Extracted {len(viols)} compliance violation(s)')
    return viols

Loading pre-trained YOLOv8 architecture weights...


## Severity Evaluation, Escalation & Immutable Auditing

In [6]:
# CELL 6 — Escalation Rules Matrix
SEV_MATRIX = {
    'pedestrian_movement':   {'y':'HIGH',    'n':'MEDIUM'},
    'equipment_intervention':{'y':'CRITICAL','n':'HIGH'},
    'electrical_panel':      {'y':'HIGH',    'n':'LOW'},
    'forklift_load':         {'y':'CRITICAL','n':'HIGH'}
}

def classify(v):
    has_person = any(w in v.get('event_description','').lower() for w in ('person','personnel'))
    m = SEV_MATRIX.get(v.get('behavior_class',''), {})
    sev = m.get('y' if has_person else 'n', 'MEDIUM')
    if v.get('_callout') == 'CRITICAL SAFETY NOTICE' and sev == 'MEDIUM': 
        sev = 'HIGH'
    return sev

def escalate(v):
    sev = v.get('severity','LOW')
    insert_violation(v)
    if sev in ('HIGH', 'CRITICAL'):
        # Route high-priority events instantly to real-time notification streams
        with open(OUT / 'alerts.jsonl', 'a') as f: 
            f.write(json.dumps({k:v[k] for k in AUDIT_FIELDS if k in v}) + "\n")
        return 'Real-time alert triggered + DB log'
    return 'Logged to DB'

def save_report(v):
    rec = {f:v.get(f,'') for f in AUDIT_FIELDS}
    with open(OUT / 'compliance_log.jsonl', 'a') as f: 
        f.write(json.dumps(rec) + "\n")
        
    cp = OUT / 'compliance_log.csv'
    hdr = not cp.exists()
    with open(cp, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=AUDIT_FIELDS)
        if hdr: 
            w.writeheader()
        w.writerow(rec)

def run_pipeline(path):
    viols = process_clip(path)
    for v in viols:
        v['severity']          = classify(v)
        v['escalation_action'] = escalate(v)
        save_report(v)
        print(f"  {SEV_ICONS.get(v['severity'],'?')} [{v['severity']}] {v['behavior_class']} | {v['zone']}")
    return viols

## Operations Dashboard Interface

In [7]:
# CELL 7 — Interactive Operation Terminal Dashboard
def _badge(sev):
    c = SEV_COLORS.get(sev, '#999')
    return f'<span style="background:{c};color:#000;padding:2px 8px;border-radius:4px;font-weight:bold">{sev}</span>'

out_a = widgets.Output()
def show_live(_=None):
    with out_a:
        clear_output()
        rows = get_violations()[:10]
        crit = [r for r in rows if r.get('severity') in ('HIGH','CRITICAL')]
        if crit:
            a = crit[0]
            display(HTML(f'<div style="background:#dc3545;color:#fff;padding:10px;border-radius:6px;margin-bottom:8px">🚨 <b>CRITICAL ALARM TRIGGERED [{a["severity"]}]</b> — {a["behavior_class"]} | Location: {a["zone"]}</div>'))
        if not rows: 
            display(HTML('<p>Waiting for surveillance streams processing...</p>'))
            return
        for v in rows:
            display(HTML(f'<div style="border:1px solid #ddd;padding:8px;margin:3px;border-radius:4px">{_badge(v["severity"])} <b>{v["behavior_class"].replace("_"," ").title()}</b> | Zone: {v["zone"]} | Timestamp: {v["timestamp"][:19]}<br><small>{v["event_description"]}</small></div>'))

b_a = widgets.Button(description='🔄 Refresh Monitor', button_style='info')
b_a.on_click(show_live)
tab_a = widgets.VBox([widgets.HTML('<h3>📹 Real-Time Live Feed Target Monitor</h3>'), b_a, out_a])

out_b = widgets.Output()
def show_timeline(_=None):
    with out_b:
        clear_output()
        rows = get_violations()
        if not rows: 
            display(HTML('<p>No recorded incidents logged.</p>'))
            return
        h = '<table style="width:100%;border-collapse:collapse;text-align:left"><tr style="background:#f2f2f2;height:30px"><th>Severity</th><th>Timestamp</th><th>Violation Classification Domain</th><th>Source Clip ID</th></tr>'
        for v in rows:
            h += f'<tr style="border-bottom:1px solid #eee;height:35px"><td>{_badge(v["severity"])}</td><td>{v["timestamp"][:19]}</td><td>{v["behavior_class"].replace("_"," ").title()}</td><td>{v["clip_id"]}</td></tr>'
        display(HTML(h + '</table>'))

b_b = widgets.Button(description='🔄 Update Timeline', button_style='info')
b_b.on_click(show_timeline)
tab_b = widgets.VBox([widgets.HTML('<h3>🔔 Incident Alert Timeline Log Stream</h3>'), b_b, out_b])

dd_sev = widgets.Dropdown(options=['All'] + SEV_ORDER, description='Risk Tier:')
dd_cls = widgets.Dropdown(options=['All'] + BEHAVIORS, description='Category:')
out_c = widgets.Output()

def show_log(_=None):
    with out_c:
        clear_output()
        f = {}
        if dd_sev.value != 'All': f['severity'] = dd_sev.value
        if dd_cls.value != 'All': f['behavior_class'] = dd_cls.value
        rows = get_violations(f)
        display(HTML(f'<b>Found {len(rows)} database record entry match results</b>'))
        if rows: 
            display(pd.DataFrame(rows)[AUDIT_FIELDS])
        else: 
            display(HTML('<p>No audit database logs found corresponding to current filtering.</p>'))

b_c = widgets.Button(description='🔍 Filter Records', button_style='primary')
b_c.on_click(show_log)
tab_c = widgets.VBox([widgets.HTML('<h3>📋 Compliance Historical Log & System Export Terminal</h3>'), widgets.HBox([dd_sev, dd_cls, b_c]), out_c])

dash = widgets.Tab(children=[tab_a, tab_b, tab_c])
for i, t in enumerate(['📹 Live Feed Monitor', '🔔 Timeline Stream', '📋 Log Historical Audit']): 
    dash.set_title(i, t)

display(dash)
show_live()

### Run Entire Pipeline Over Downloaded Video Clips

In [8]:
# CELL 8 — EXECUTE PIPELINE OVER EVALUATION DATASET CLIPS
video_clips = list(DATA.glob('*.mp4')) + list(DATA.glob('*.avi'))

if not video_clips:
    print(f"❌ No clip found inside local folder pathway target: '{DATA}'. Please ensure Cell 2 successfully ran.")
else:
    print(f"🚀 Found {len(video_clips)} videos to process. Initializing compliance scanning run...\n")
    for i, clip_path in enumerate(video_clips):
        print(f"[{i+1}/{len(video_clips)}] Processing source asset stream...")
        run_pipeline(str(clip_path))
        
    print('\n🏁 All evaluation clips processed successfully. Running dashboard engine view sync...')
    show_live()

🚀 Found 691 videos to process. Initializing compliance scanning run...

[1/691] Processing source asset stream...
🎥 Evaluated file [1_tr42]: Extracted 4 compliance violation(s)
  🟠 [HIGH] pedestrian_movement | Zone-3
  🔴 [CRITICAL] forklift_load | Zone-Forklift
  🔴 [CRITICAL] forklift_load | Zone-Forklift
  🔴 [CRITICAL] forklift_load | Zone-Forklift
[2/691] Processing source asset stream...
🎥 Evaluated file [3_tr31]: Extracted 1 compliance violation(s)
  🟠 [HIGH] pedestrian_movement | Zone-3
[3/691] Processing source asset stream...
🎥 Evaluated file [0_tr155]: Extracted 3 compliance violation(s)
  🟠 [HIGH] pedestrian_movement | Zone-3
  🟠 [HIGH] pedestrian_movement | Zone-3
  🟠 [HIGH] pedestrian_movement | Zone-3
[4/691] Processing source asset stream...
🎥 Evaluated file [4_tr36]: Extracted 3 compliance violation(s)
  🟠 [HIGH] pedestrian_movement | Zone-3
  🟠 [HIGH] pedestrian_movement | Zone-3
  🟠 [HIGH] pedestrian_movement | Zone-3
[5/691] Processing source asset stream...
🎥 Evaluate